# Tractogram → Zarr Vectors: Import, Pyramid, Inspect

This notebook demonstrates:
1. Generating a synthetic tractogram (or loading a real `.trk` file)
2. Writing it to a zarr vectors store with supervoxel binning
3. Building a multi-level pyramid with object sparsity
4. Inspecting the on-disk store structure

In [12]:
import numpy as np
from pathlib import Path
import json

# Adjust this path if you have zarr-vectors installed from source
import sys
sys.path.insert(0, '.')  # if running from the repo root

## 1. Create or Load a Tractogram

**Option A** — Synthetic streamlines (no dependencies):

In [13]:
rng = np.random.default_rng(42)
n_streamlines = 500

streamlines = []
for i in range(n_streamlines):
    n_pts = rng.integers(20, 80)
    start = rng.uniform(10, 190, size=(1, 3)).astype(np.float32)
    steps = rng.normal(0, 2.5, size=(n_pts - 1, 3)).astype(np.float32)
    trajectory = np.concatenate([start, start + np.cumsum(steps, axis=0)])
    trajectory = np.clip(trajectory, 0, 199).astype(np.float32)
    streamlines.append(trajectory)

print(f"Generated {len(streamlines)} streamlines")
print(f"Vertices per streamline: {min(len(s) for s in streamlines)}–{max(len(s) for s in streamlines)}")
print(f"Total vertices: {sum(len(s) for s in streamlines)}")

Generated 500 streamlines
Vertices per streamline: 20–79
Total vertices: 25042


**Option B** — Load a real `.trk` file (requires `nibabel`):

```python
import nibabel as nib

trk = nib.streamlines.load("my_tracts.trk")
streamlines = [np.asarray(s, dtype=np.float32) for s in trk.streamlines]
print(f"Loaded {len(streamlines)} streamlines from TRK")
```

## 2. Write to Zarr Vectors Store

We use `chunk_shape` for disk partitioning and `bin_shape` for
supervoxel vertex grouping within each chunk.

In [14]:
from zarr_vectors.types.polylines import write_polylines

store_path = "tracts.zarrvectors"

# Assign streamlines to two groups (e.g. left/right hemisphere)
groups = {
    0: list(range(0, 250)),     # "left hemisphere"
    1: list(range(250, 500)),   # "right hemisphere"
}

summary = write_polylines(
    store_path,
    streamlines,
    chunk_shape=(100.0, 100.0, 100.0),   # 8 chunks covering 200³ space
    bin_shape=(10.0, 10.0, 10.0),         # 2³=8 bins per chunk
    groups=groups,
)

print(f"Written: {summary['polyline_count']} streamlines")
print(f"         {summary['vertex_count']} total vertices")
print(f"         {summary['chunk_count']} spatial chunks")
print(f"         {summary['group_count']} groups")
print(f"         {summary['cross_chunk_link_count']} cross-chunk links")

Written: 500 streamlines
         25042 total vertices
         8 spatial chunks
         2 groups
         825 cross-chunk links


## 3. Build Multi-Resolution Pyramid

Create two coarser levels:
- **Level 1**: 2× bin ratio (8× vertex reduction), keep all streamlines
- **Level 2**: 2× bin ratio (8× vertex reduction), keep 50% of streamlines (16× total)

In [15]:
from zarr_vectors.multiresolution.coarsen import build_pyramid

pyramid = build_pyramid(
    store_path,
    level_configs=[
        {"bin_ratio": (2, 2, 2), "object_sparsity": 1.0},
        {"bin_ratio": (2, 2, 2), "object_sparsity": 0.5},
    ],
)

print(f"Pyramid: {pyramid['levels_created']} levels created")
for spec in pyramid['level_specs']:
    print(f"  Level {spec['level']}: "
          f"bin_ratio={spec.get('bin_ratio')}, "
          f"sparsity={spec.get('object_sparsity')}, "
          f"volume_reduction={spec.get('expected_volume_reduction')}×")

Pyramid: 1 levels created
  Level 1: bin_ratio=[2, 2, 2], sparsity=1.0, volume_reduction=8.0×


## 4. Inspect the Store

### 4a. Store metadata

In [16]:
from zarr_vectors.core.store import open_store, read_root_metadata, list_resolution_levels, read_level_metadata

root = open_store(store_path)
meta = read_root_metadata(root)

print("=== Root Metadata ===")
print(f"  Format version:  {meta.format_version}")
print(f"  Geometry types:  {meta.geometry_types}")
print(f"  Chunk shape:     {meta.chunk_shape}")
print(f"  Bin shape:       {meta.effective_bin_shape}")
print(f"  Bins per chunk:  {meta.bins_per_chunk}")
print(f"  Bounds:          {meta.bounds}")
print(f"  Dimensions:      {meta.sid_ndim}D")
print()

levels = list_resolution_levels(root)
print(f"=== {len(levels)} Resolution Levels ===")
for li in levels:
    lm = read_level_metadata(root, li)
    print(f"  Level {li}:")
    print(f"    vertex_count:     {lm.vertex_count}")
    print(f"    bin_shape:        {lm.bin_shape}")
    print(f"    bin_ratio:        {lm.bin_ratio}")
    print(f"    object_sparsity:  {lm.object_sparsity}")
    print(f"    coarsening:       {lm.coarsening_method}")

=== Root Metadata ===
  Format version:  0.2
  Geometry types:  ['streamline']
  Chunk shape:     (100.0, 100.0, 100.0)
  Bin shape:       (10.0, 10.0, 10.0)
  Bins per chunk:  (10, 10, 10)
  Bounds:          ([0.0, 0.0, 0.0], [199.0, 199.0, 199.0])
  Dimensions:      3D

=== 2 Resolution Levels ===
  Level 0:
    vertex_count:     25042
    bin_shape:        None
    bin_ratio:        None
    object_sparsity:  1.0
    coarsening:       none
  Level 1:
    vertex_count:     913
    bin_shape:        (20.0, 20.0, 20.0)
    bin_ratio:        (2, 2, 2)
    object_sparsity:  1.0
    coarsening:       grid_metanode


### 4b. Directory tree on disk

In [17]:
def print_tree(path, prefix="", max_depth=3, _depth=0):
    """Print a directory tree with file sizes."""
    path = Path(path)
    if _depth > max_depth:
        return

    entries = sorted(path.iterdir(), key=lambda p: (p.is_file(), p.name))
    for i, entry in enumerate(entries):
        is_last = i == len(entries) - 1
        connector = "└── " if is_last else "├── "
        if entry.name.startswith("__"):
            continue

        if entry.is_file():
            size = entry.stat().st_size
            if size < 1024:
                size_str = f"{size} B"
            elif size < 1024 * 1024:
                size_str = f"{size / 1024:.1f} KB"
            else:
                size_str = f"{size / (1024*1024):.2f} MB"
            print(f"{prefix}{connector}{entry.name}  ({size_str})")
        else:
            n_children = sum(1 for _ in entry.iterdir() if not _.name.startswith("__"))
            print(f"{prefix}{connector}{entry.name}/  [{n_children} items]")
            extension = "    " if is_last else "│   "
            print_tree(entry, prefix + extension, max_depth, _depth + 1)

print(f"\n=== Store: {store_path} ===")
print_tree(store_path, max_depth=4)


=== Store: tracts.zarrvectors ===
├── parametric/  [0 items]
├── resolution_0/  [6 items]
│   ├── cross_chunk_links/  [2 items]
│   │   ├── .zattrs  (75 B)
│   │   └── data  (51.6 KB)
│   ├── groupings/  [3 items]
│   │   ├── .zattrs  (49 B)
│   │   ├── data  (3.9 KB)
│   │   └── offsets  (16 B)
│   ├── object_index/  [3 items]
│   │   ├── .zattrs  (72 B)
│   │   ├── data  (381.1 KB)
│   │   └── offsets  (3.9 KB)
│   ├── vertex_group_offsets/  [8 items]
│   │   ├── 0.0.0  (20.1 KB)
│   │   ├── 0.0.1  (25.5 KB)
│   │   ├── 0.1.0  (29.5 KB)
│   │   ├── 0.1.1  (25.8 KB)
│   │   ├── 1.0.0  (21.0 KB)
│   │   ├── 1.0.1  (18.7 KB)
│   │   ├── 1.1.0  (26.6 KB)
│   │   └── 1.1.1  (23.4 KB)
│   ├── vertices/  [9 items]
│   │   ├── .zattrs  (72 B)
│   │   ├── 0.0.0  (31.7 KB)
│   │   ├── 0.0.1  (39.9 KB)
│   │   ├── 0.1.0  (45.5 KB)
│   │   ├── 0.1.1  (39.4 KB)
│   │   ├── 1.0.0  (32.1 KB)
│   │   ├── 1.0.1  (28.9 KB)
│   │   ├── 1.1.0  (39.7 KB)
│   │   └── 1.1.1  (36.3 KB)
│   └── .zattrs  (27

### 4c. Per-level chunk statistics

In [18]:
from zarr_vectors.core.arrays import list_chunk_keys, read_chunk_vertices
from zarr_vectors.core.store import get_resolution_level

for li in levels:
    lg = get_resolution_level(root, li)
    chunk_keys = list_chunk_keys(lg)
    
    verts_per_chunk = []
    vgs_per_chunk = []
    for ck in chunk_keys:
        try:
            groups = read_chunk_vertices(lg, ck, dtype=np.float32, ndim=3)
            vgs_per_chunk.append(len(groups))
            verts_per_chunk.append(sum(len(g) for g in groups))
        except Exception:
            pass
    
    if verts_per_chunk:
        print(f"Level {li}: {len(chunk_keys)} chunks")
        print(f"  Vertices: total={sum(verts_per_chunk)}, "
              f"per-chunk min={min(verts_per_chunk)} / "
              f"max={max(verts_per_chunk)} / "
              f"mean={np.mean(verts_per_chunk):.0f}")
        print(f"  VGs/chunk: min={min(vgs_per_chunk)} / "
              f"max={max(vgs_per_chunk)} / "
              f"mean={np.mean(vgs_per_chunk):.0f}")

Level 0: 8 chunks
  Vertices: total=25042, per-chunk min=2467 / max=3879 / mean=3130
  VGs/chunk: min=1199 / max=1891 / mean=1524
Level 1: 8 chunks
  Vertices: total=913, per-chunk min=106 / max=123 / mean=114
  VGs/chunk: min=1 / max=1 / mean=1


### 4d. Validate the store

In [19]:
from zarr_vectors.validate import validate

result = validate(store_path, level=5)
print(result.summary())
if result.warnings:
    for w in result.warnings:
        print(f"  WARN: {w}")

Level 5 validation: PASS
  32 passed, 0 warnings, 0 errors


### 4e. Read back and verify

In [20]:
from zarr_vectors.types.polylines import read_polylines

# Read all streamlines at full resolution
result_l0 = read_polylines(store_path, level=0)
print(f"Level 0: {result_l0['polyline_count']} streamlines, {result_l0['vertex_count']} vertices")

# Read by group
result_g0 = read_polylines(store_path, group_ids=[0])
print(f"Group 0: {result_g0['polyline_count']} streamlines")

# Read single streamline
result_s = read_polylines(store_path, object_ids=[42])
print(f"Streamline 42: {result_s['vertex_count']} vertices")

Level 0: 500 streamlines, 25042 vertices
Group 0: 250 streamlines
Streamline 42: 40 vertices


### 4f. Lazy access (optional — requires dask)

In [21]:
from zarr_vectors.lazy import open_zvr

zvr = open_zvr(store_path)
print(zvr)
print(f"Level 0: {zvr[0]}")
print(f"Polylines: {zvr[0].polylines.count}")

# Lazy access to a single streamline
poly_42 = zvr[0].polylines[42].compute()
print(f"Streamline 42: {poly_42.shape}")

# Filter by group
view = zvr[0].filter(group_ids=[0])
print(f"Group 0 view: {view}")

ModuleNotFoundError: No module named 'zarr_vectors.lazy'

## 5. Cleanup

In [ ]:
import shutil
shutil.rmtree(store_path, ignore_errors=True)
print("Store removed.")